<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

# Ejercicios de parcial

## Hechos en clase

### EJERCICIO Nº1
Realizar una consulta SQL que devuelva todos los clientes que durante 2 años consecutivos compraron al menos 5 productos distintos. De esos clientes mostrar:
- El código de cliente
- El monto total comprado en el 2012
- La cantidad de unidades de productos compradas en el 2012

El resultado debe ser ordenado primero por aquellos clientes que compraron solo productos compuestos en algún momento, luego el resto.

In [ ]:
-- EJERCICIO Nº1 (misma solución que 05-EjerciciosSueltos/SolucionesParcial.sql, parte 1). Resultado en esta base: 34 clientes.
SELECT
    c.clie_codigo,
    ( SELECT ISNULL(SUM(f.fact_total), 0)
      FROM Factura f
      WHERE f.fact_cliente = c.clie_codigo AND YEAR(f.fact_fecha) = 2012 )          AS monto_2012,
    ( SELECT ISNULL(SUM(i.item_cantidad), 0)
      FROM Factura f
      JOIN Item_Factura i ON i.item_tipo = f.fact_tipo AND i.item_sucursal = f.fact_sucursal AND i.item_numero = f.fact_numero
      WHERE f.fact_cliente = c.clie_codigo AND YEAR(f.fact_fecha) = 2012 )          AS unidades_2012
FROM Cliente c
WHERE EXISTS (                                   -- "existe un AÑO ANCLA tal que..."
    SELECT 1
    FROM Factura f1
    JOIN Item_Factura i1 ON i1.item_tipo = f1.fact_tipo AND i1.item_sucursal = f1.fact_sucursal AND i1.item_numero = f1.fact_numero
    WHERE f1.fact_cliente = c.clie_codigo
    GROUP BY YEAR(f1.fact_fecha)                 -- un grupo por año = candidato a ancla
    HAVING COUNT(DISTINCT i1.item_producto) >= 5 -- ...en el año ancla compró >= 5 distintos
       AND ( SELECT COUNT(DISTINCT i2.item_producto)
             FROM Factura f2
             JOIN Item_Factura i2 ON i2.item_tipo = f2.fact_tipo AND i2.item_sucursal = f2.fact_sucursal AND i2.item_numero = f2.fact_numero
             WHERE f2.fact_cliente = c.clie_codigo
               AND YEAR(f2.fact_fecha) = YEAR(f1.fact_fecha) + 1 ) >= 5   -- ...y en el año siguiente también
)
ORDER BY
    CASE WHEN EXISTS ( SELECT 1                                   -- existe una factura del cliente...
                       FROM Factura f
                       WHERE f.fact_cliente = c.clie_codigo
                         AND NOT EXISTS ( SELECT 1                 -- ...en la que NO hay ningún item...
                                          FROM Item_Factura i
                                          WHERE i.item_tipo = f.fact_tipo AND i.item_sucursal = f.fact_sucursal AND i.item_numero = f.fact_numero
                                            AND i.item_producto NOT IN ( SELECT comp_producto FROM Composicion ) ) )  -- ...que NO sea compuesto
         THEN 0 ELSE 1 END,
    c.clie_codigo;